# SecureMed Model Selection

Staged evaluation: GridSearchCV -> FHE simulate (all configs) -> FHE execute (top 5 finalists).

See also `scripts/run_model_selection.py` for a non-interactive run that writes the same artifacts.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

REPO_ROOT = Path('..').resolve()
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / 'scripts'))

from model_selection_utils import (
    MODELS_DIR,
    load_splits,
    run_staged_selection,
    MODEL_FAMILIES,
    save_cleartext_baselines,
    save_selected_models,
)

In [ ]:
df_train, df_test, X_train, y_train, X_test, y_test = load_splits()
sample = df_test.sample(1, random_state=8047).drop(columns=['prognosis', 'prognosis_encoded']).values
df_train['prognosis'].value_counts().head(10).plot(kind='bar', figsize=(10, 4), title='Top disease frequencies')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
selections = {}
all_results = {}
for key in ('logistic_regression', 'xgboost'):
    family = MODEL_FAMILIES[key]
    results, selection = run_staged_selection(
        family, X_train, y_train, X_test, y_test, sample, models_dir=MODELS_DIR
    )
    all_results[key] = results
    selections[key] = selection

save_selected_models(selections)
save_cleartext_baselines(X_train, y_train, X_test, y_test, selections)

In [ ]:
for key, df in all_results.items():
    if 'fhe_execute_time_s' not in df.columns:
        continue
    finalists = df[df['fhe_execute_time_s'].notna()].head(5)
    plt.figure(figsize=(6, 4))
    plt.scatter(finalists['fhe_execute_time_s'], finalists['mean_test_score'])
    plt.xlabel('FHE execute time (s)')
    plt.ylabel('CV accuracy')
    plt.title(f'{key}: accuracy vs FHE latency (finalists)')
    plt.tight_layout()
    plt.show()

## Discussion

1. **Grid ranges** follow the proposal (`max_depth <= 3`, `n_estimators <= 30`) and Concrete-ML guidance that lower `n_bits` reduces ciphertext size and FHE latency.
2. **CV-best != FHE-best**: cross-validation accuracy alone is insufficient; we measure FHE simulate on all configs and FHE execute on finalists.
3. **LR `n_bits` retuning** targets the 1 MB ciphertext threshold while keeping accuracy delta <= 5 pp.
4. **Dual UI**: both models are deployed so users can compare linear vs tree-ensemble FHE behavior on the same symptom workflow.